# Quantonium — Wan generation training
This notebook fine-tunes **Wan 2.1 T2V 1.3B** with LoRA, then generates videos using the saved adapter. The pretrained base stays frozen; the adapters change the generator's behavior. This is a separate training lineage from your V0.8 interpolator.

**Run sections 1–3 on your L4; leave section 3 running.** First launch downloads the pretrained Wan components and prepares cached training inputs. Later launches resume saved Wan weights, optimizer and random state. Your V0.8 checkpoint at phase 19,369 stays in OpenV08.

Default: continuous updates, save verification every 25 updates, rank 8, 256×256, 17-frame motion clips, batch 1 with 4 accumulated microbatches. These are conservative starting settings; full-size L4 fit and generation quality have not been tested in the authoring workspace. A real GPU update check must pass before the loop proceeds.

Eight original JC cels teach **appearance as individual stills**; Big Buck Bunny clips teach motion; Sintel is held out. This small pilot cannot establish general animation ability or exact character identity. Generated/interpolated JC videos are not training targets.

Press Stop once in section 3; wait for `WAN STOPPED` and a verified save. Then run sections 4–5 to inspect progress and generate. Colab must remain alive to train. Reconnect and rerun 1–3 after a runtime loss. No external paid job is started.

## 1. Mount Drive, install and check code
Keep the V0.8 training cell stopped while Wan uses the GPU. This uses a separate Python environment with Colab's existing CUDA PyTorch. Large pretrained downloads happen once per runtime; expect multi-GB storage and an initial preparation delay.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, shutil, time, fcntl
import torch
from google.colab import drive
drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select a GPU runtime.'
assert torch.cuda.is_bf16_supported(), 'Use L4/A100; the T4/FP16 path is not validated.'
print('GPU:', torch.cuda.get_device_name(0), '| GiB:', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1))
CODE_COMMIT = '0841dbfc9f7bfc4c0ba5bb70e264016dfd31d7bf'
DRIVE = Path('/content/drive/MyDrive/Quantonium/Video/Wan21_LoRA')
DATA = DRIVE/'dataset_256_17_v1'
RUN = DRIVE/'run_v1'
TARGET_WAN_STEP = 0  # Continuous. A positive value means total Wan optimizer updates.
SAVE_EVERY = 25
STOP = Path('/content/WAN_STOP_REQUEST')

def assert_idle():
    for path in [Path('/content/quantonium_open_video/run/training.lock'), RUN/'training.lock']:
        if path.exists():
            with path.open('a') as handle:
                try:
                    fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)
                except BlockingIOError:
                    raise RuntimeError('A video trainer is still running: '+str(path))
assert_idle()
DRIVE.mkdir(parents=True, exist_ok=True)
REPO = Path('/content')/('quantonium_wan_'+CODE_COMMIT[:10])
if not (REPO/'.git').exists():
    subprocess.run(['git','init',str(REPO)],check=True)
    subprocess.run(['git','-C',str(REPO),'remote','add','origin','https://github.com/Anharmoniclabs/quantoniumoslinux.git'],check=True)
    subprocess.run(['git','-C',str(REPO),'sparse-checkout','set','runtime/experiments/quantonium_video'],check=True)
    subprocess.run(['git','-C',str(REPO),'fetch','--depth=1','origin',CODE_COMMIT],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','--detach','FETCH_HEAD'],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==CODE_COMMIT
LAB = REPO/'runtime/experiments/quantonium_video'
VENV = Path('/content/wan_training_env')
if not (VENV/'bin/python').exists():
    subprocess.run([sys.executable,'-m','venv','--system-site-packages',str(VENV)],check=True)
PYTHON = str(VENV/'bin/python')
constraint=Path('/content/wan_torch_constraint.txt')
constraint.write_text('torch=='+str(torch.__version__)+'\n')
subprocess.run([PYTHON,'-m','pip','install','-q','-c',str(constraint),'-r',str(LAB/'requirements-wan-training.txt')],check=True)
if not shutil.which('ffmpeg'):
    subprocess.run(['apt-get','-qq','update'],check=True)
    subprocess.run(['apt-get','-qq','install','-y','ffmpeg'],check=True)
os.environ['TOKENIZERS_PARALLELISM']='false'
# No pretrained downloads for these real tiny-Wan gradient/resume/inference tests.
subprocess.run([PYTHON,'-m','unittest','test_wan_training','-v'],cwd=LAB,check=True)
print('WAN SOFTWARE CHECKS PASSED. Full pretrained GPU check occurs in section 3.')

## 2. Prepare Wan's training inputs
Downloads the checksummed Blender films; takes 17 consecutive frames per accepted clip. The cut filter is a heuristic. The eight approved JC pose files are already in the repository and checked against their recorded hashes.

The text encoder runs first and unloads; the VAE then caches normalized video latents and unloads. Training keeps neither large encoder in GPU memory. The completed cache and source records live in Drive; subsequent runs verify and reuse them. Review the contact sheet below; it shows each accepted sample's middle frame. Film captions are generic, so this is an initial adaptation dataset.

In [ ]:
assert_idle()
subprocess.run([PYTHON,'-u',str(LAB/'prepare_wan_training.py'),
    '--out',str(DATA),'--movies','/content/wan_open_movies',
    '--size','256','--frames','17','--clips','64'],cwd=LAB,check=True)
from IPython.display import display, Image
info=json.loads((DATA/'manifest.json').read_text())
print('Cached samples:', {g:sum(r['group']==g for r in info['records']) for g in ['appearance','motion','validation']})
display(Image(filename=str(DATA/'data_review.jpg')))

## 3. Start / resume continuous Wan training
**This cell updates the generator's LoRA weights.** `WAN GPU UPDATE VERIFIED` means a real gradient changed the adapters. `WAN DRIVE SNAPSHOT VERIFIED N` means the adapter, optimizer and random states at step N were written and read back successfully.

Wan has its own step count starting at 0. Your 19,369 V0.8 updates cannot be loaded into Wan's different architecture. One optimizer update here accumulates four examples, sampled equally between JC appearance and real video.

Every 25 updates the trainer evaluates held-out film latents and saves to Drive. The first GPU update also saves immediately. It retains the **latest three complete snapshots, best validation snapshot and step zero** to bound storage. Original V0.8 files are never rotated. Best is selected by fixed latent flow loss, which does not establish visual quality and is not comparable to V0.8 L1.

Leave this cell active. Press Stop once and wait for `WAN STOPPED`. Do not change the dataset, rank or optimizer settings to resume this run; use a new DATA/RUN folder for a different experiment.

In [ ]:
assert_idle()
assert (DATA/'manifest.json').exists(), 'Run section 2 first.'
STOP.unlink(missing_ok=True)
RUN.mkdir(parents=True, exist_ok=True)
logs=DRIVE/'logs'; logs.mkdir(exist_ok=True)
sys.path.insert(0,str(LAB))
from colab_video_runtime import run_foreground
command=[PYTHON,'-u',str(LAB/'train_wan_lora.py'),
    '--data',str(DATA),'--out',str(RUN),'--steps',str(TARGET_WAN_STEP),
    '--rank','8','--lr','0.0001','--accumulation','4',
    '--save-every',str(SAVE_EVERY),'--stop-file',str(STOP)]
if (RUN/'run_config.json').exists():
    command.append('--require-resume')
print('Wan target:', 'continuous' if TARGET_WAN_STEP==0 else TARGET_WAN_STEP)
print('Leave this cell running. Stop once requests a clean save.')
run_foreground(command,LAB,logs/('wan_'+time.strftime('%Y%m%d_%H%M%S')+'.log'),STOP)

## 4. Inspect saved progress
Run after section 3 has stopped. A lower loss is useful evidence of fit, but inspect generated motion, identity and anatomy before calling it an improvement.

In [ ]:
assert_idle()
for name in ['latest.json','best.json']:
    path=RUN/name
    if path.exists():
        print(name, json.dumps(json.loads(path.read_text()),indent=2))
    else:
        print(name, 'No saved Wan update yet.')

## 5. Generate using the trained Wan adapter
This loads the saved LoRA into the same pinned pretrained Wan model. Default: **latest**, so the clip uses the most recent updates. Switch to `best` for comparison. A best checkpoint at 0 means validation has not yet beaten the unadapted starting point.

The small 256×256 preview is a diagnostic. Text-to-video plus a small LoRA does not guarantee JC's exact face or pose. Set `COMPARE_BASE = True` to also render the pretrained base with the same prompt, seed and settings. Frames, MP4 and checkpoint provenance are saved to Drive. Generation runs after training stops so they do not compete for GPU memory.

In [ ]:
assert_idle()
from IPython.display import Video, display
CHECKPOINT = 'latest'  # 'latest' or 'best'
COMPARE_BASE = False
PROMPT = ('cemi_jc, original 2D anime young man with brown skin, short dark hair, '
    'huge white eyes with narrow dark pupils, gray jacket over black streetwear, '
    'headphones, gold necklace and white sneakers, full body, performs one controlled '
    'martial arts kick and lands, fixed camera, fine outlines, flat cel shading, plain light gray background')
SEED=808
previews=DRIVE/'previews'; previews.mkdir(exist_ok=True)
for mode in (['base',CHECKPOINT] if COMPARE_BASE else [CHECKPOINT]):
    out=previews/(time.strftime('%Y%m%d_%H%M%S')+'_'+mode)
    command=[PYTHON,'-u',str(LAB/'generate_wan.py'),'--prompt',PROMPT,
        '--out',str(out),'--seed',str(SEED),'--frames','17','--steps','30','--height','256','--width','256']
    if mode!='base':
        command += ['--lora-run',str(RUN),'--checkpoint',mode]
    subprocess.run(command,cwd=LAB,check=True)
    print(mode, 'Provenance:', (out/'provenance.json').read_text())
    display(Video(str(out/'generated.mp4'),embed=True))

## Journal and evidence
- Prior user-reported V0.8 result: phase 19,369; JC L1 0.007836; held-out film L1 0.0356248. These describe the interpolator, not Wan.
- Eight local Wan tests passed. Local Wan checks: actual tiny Wan transformer backward/update; frozen base unchanged; exact next-update resume with optimizer/RNG; Diffusers adapter export/load; two-step latent generation; corruption fallback; bounded snapshot retention; cache hashing and film-split rejection.
- Actual tiny Wan VAE: one-frame and 17-frame encode/decode shape checks passed on CPU.
- Full pretrained model download, L4 VRAM fit, full-size training and generated-video quality were **not executed in the authoring environment**. The Colab live outputs are the source of evidence for those.
- Default extraction produced 8 JC stills, 26 training video clips and 8 held-out clips in local preparation; middle-frame review showed no title cards. Full clips still need motion review.
- No Seedance weights are used. This is Wan LoRA adaptation integrated into the Quantonium video project; it does not merge the 87,817-parameter interpolator into Wan or provide exact Blender/image conditioning.

Sources: [Wan pretrained model](https://huggingface.co/Wan-AI/Wan2.1-T2V-1.3B-Diffusers), [Diffusers Wan API and LoRA loading](https://huggingface.co/docs/diffusers/api/pipelines/wan), [Big Buck Bunny attribution](https://peach.blender.org/about/), [Sintel attribution](https://durian.blender.org/sharing/). Revisions and source hashes are in `open_sources.json` and the cache manifest.